# simple-idp: extract a PDF, then embed it

This notebook is a minimal end-to-end example for `idp-client`:

1. Load a PDF and extract it via `IdpClient.extract(..., format="markdown")`.
2. Join the page texts into a single document.
3. Request an embedding from IDP's OpenAI-compatible `POST /v1/embeddings`
   endpoint via the public `IdpClient.embed(...)` method.

All knobs (file path, IDP base URL + key, embedding model) live in the single
configuration cell below. Each falls back to an environment variable of the
same name, so you can either edit the cell or `export` the variables before
launching Jupyter.

If `client.embed(...)` returns a 422 naming the model, query IDP's profile
registry to see which model strings it accepts:

```
GET {IDP_BASE_URL}/v1/embedding/models
```

Run all cells top-to-bottom.


In [ ]:
import os

# --- Inputs --------------------------------------------------------------
PDF_PATH = os.environ.get("PDF_PATH", "sample.pdf")

# --- IDP (idp-client) ----------------------------------------------------
IDP_BASE_URL = os.environ.get("IDP_BASE_URL", "http://localhost:8000")
IDP_API_KEY  = os.environ.get("IDP_API_KEY")

# --- Embedding -----------------------------------------------------------
# IDP serves both extraction and embedding under the same base URL + key,
# so no separate embedding URL or credential is needed.
#
# `EMBEDDING_MODEL` is an IDP **profile id**, NOT a HuggingFace model name.
# Document-side Qwen3 is `qwen3-0.6b-doc`; query-side is `qwen3-0.6b-query`.
# Other registered profiles: `bge-m3`, `nomic-v1.5-doc`, `nomic-v1.5-query`.
# List the live registry with: GET {IDP_BASE_URL}/v1/embedding/models
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "qwen3-0.6b-doc")


In [2]:
from pathlib import Path

if not IDP_API_KEY:
    raise RuntimeError(
        "Missing required configuration value(s): IDP_API_KEY. "
        "Set it in the configuration cell above or export the env var."
    )

if not Path(PDF_PATH).is_file():
    raise FileNotFoundError(PDF_PATH)


In [3]:
from idp_client import IdpClient

with IdpClient(base_url=IDP_BASE_URL, api_key=IDP_API_KEY) as client:
    result = client.extract(PDF_PATH, format="markdown")
    text = "\n\n".join(result.pages)
    print(f"pages={len(result.pages)} chars={len(text)}")
    response = client.embed(text, model=EMBEDDING_MODEL)

embedding = response.data[0].embedding


pages=1 chars=1976


In [4]:
print(f"dim={len(embedding)}")
print(embedding[:8])


dim=1024
[-0.10801642388105392, 0.008321266621351242, -0.01276478637009859, 0.04798664152622223, 0.010836993344128132, 0.06565015763044357, 0.014478182420134544, -0.08576121926307678]
